# 04 | Export and PostgreSQL database

1. **Export** the five tables to CSV and Parquet, for Power BI.
2. **Load** them into PostgreSQL, in the schema `sales` of the database `geo_reference` (the same database as the reference project). The stores are linked to the real postal codes with a foreign key to `reference.postal_codes`, joining business data and geography in one model.
3. **Example queries**: growth with window functions, revenue by district, promotion effect, and a spatial analysis of the population around each store.

*Step 2 needs PostgreSQL + PostGIS and notebook 08 of the reference project already run. If the reference schema does not exist, the link to the postal codes is skipped.*

## Setup

In [1]:
import sys
from pathlib import Path

# Make the src/ folder importable when running from the notebooks/ folder
sys.path.append(str(Path.cwd().parent))

import numpy as np
import pandas as pd

from src.config import *
from src.utils import *
import io
import os
from getpass import getpass

from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL

tables = {name: load_table(name) for name in ["dim_store", "dim_product", "dim_date", "dim_promotion", "fact_sales"]}
for name, df in tables.items():
    print(f"{name}: {len(df):,} rows")

dim_store: 52 rows
dim_product: 33 rows
dim_date: 1,826 rows
dim_promotion: 25 rows
fact_sales: 1,504,031 rows


## 1. Export (CSV and Parquet)

Parquet is much smaller and faster than CSV, and Power BI reads it directly (**Get data → Parquet**).

In [2]:
for folder in ["csv", "parquet"]:
    (OUTPUTS_DIR / folder).mkdir(parents=True, exist_ok=True)

for name, df in tables.items():
    df.to_csv(OUTPUTS_DIR / "csv" / f"{name}.csv", index=False)
    df.to_parquet(OUTPUTS_DIR / "parquet" / f"{name}.parquet", index=False)

sizes = pd.DataFrame([
    {"table": name,
     "csv_MB": round((OUTPUTS_DIR / "csv" / f"{name}.csv").stat().st_size / 1e6, 2),
     "parquet_MB": round((OUTPUTS_DIR / "parquet" / f"{name}.parquet").stat().st_size / 1e6, 2)}
    for name in tables
])
sizes

,table,csv_MB,parquet_MB
0,dim_store,0.01,0.01
1,dim_product,0.00,0.01
2,dim_date,0.13,0.03
3,dim_promotion,0.00,0.01
4,fact_sales,73.43,7.33


## 2. Load into PostgreSQL

### Connection
Same database as the reference project. The password is read from the `PGPASSWORD` environment variable or asked when the cell runs.

In [3]:
DB_HOST = "localhost"
DB_PORT = 5432
DB_USER = "postgres"
DB_NAME = "geo_reference"
DB_PASSWORD = os.environ.get("PGPASSWORD") or getpass("PostgreSQL password: ")

engine = create_engine(URL.create("postgresql+psycopg2", username=DB_USER, password=DB_PASSWORD,
                                  host=DB_HOST, port=DB_PORT, database=DB_NAME))


def run_query(query, params=None):
    return pd.read_sql(text(query), engine, params=params)


def run_statements(sql):
    with engine.begin() as conn:
        conn.exec_driver_sql(sql)


def copy_to_table(df, table):
    # Bulk load with PostgreSQL COPY (much faster than INSERT for millions of rows)
    buffer = io.StringIO()
    df.to_csv(buffer, index=False, header=False, na_rep="\\N")
    buffer.seek(0)
    columns = ", ".join(df.columns)
    with engine.begin() as conn:
        cursor = conn.connection.cursor()
        cursor.copy_expert(f"COPY {table} ({columns}) FROM STDIN WITH (FORMAT csv, NULL '\\N')", buffer)


reference_exists = run_query("SELECT COUNT(*) AS n FROM information_schema.tables "
                             "WHERE table_schema = 'reference' AND table_name = 'postal_codes'")["n"].iloc[0] == 1
print("Reference schema found:", reference_exists)

Reference schema found: True


### Create the tables and load the data

In [4]:
run_statements((SQL_DIR / "01_create_tables.sql").read_text())

load_order = ["dim_store", "dim_product", "dim_date", "dim_promotion", "fact_sales"]
for name in load_order:
    df = tables[name].copy()
    for column in df.select_dtypes(include="datetime").columns:
        df[column] = df[column].dt.strftime("%Y-%m-%d")
    copy_to_table(df, f"sales.{name}")
    print(f"Loaded sales.{name}: {len(df):,} rows")

# Store location as a PostGIS point
run_statements("UPDATE sales.dim_store SET geometry = ST_SetSRID(ST_MakePoint(lon, lat), 4326);")

Loaded sales.dim_store: 52 rows
Loaded sales.dim_product: 33 rows
Loaded sales.dim_date: 1,826 rows
Loaded sales.dim_promotion: 25 rows
Loaded sales.fact_sales: 1,504,031 rows


### Keys, indexes and link to the geography

`sql/02_constraints.sql` adds the primary and foreign keys of the star schema. If the reference schema exists, `dim_store.postal_code` also becomes a foreign key to `reference.postal_codes`.

In [5]:
run_statements((SQL_DIR / "02_constraints.sql").read_text())

if reference_exists:
    run_statements("""
        ALTER TABLE sales.dim_store
            ADD CONSTRAINT fk_store_postal_code
            FOREIGN KEY (postal_code) REFERENCES reference.postal_codes (postal_code);
    """)
    print("Stores linked to reference.postal_codes")

run_query("""
    SELECT table_schema, table_name, constraint_type, constraint_name
    FROM information_schema.table_constraints
    WHERE table_schema = 'sales' AND constraint_type IN ('PRIMARY KEY', 'FOREIGN KEY')
    ORDER BY constraint_type DESC, table_name
""")

Stores linked to reference.postal_codes


,table_schema,table_name,constraint_type,constraint_name
0,sales,dim_date,PRIMARY KEY,pk_dim_date
1,sales,dim_product,PRIMARY KEY,pk_dim_product
2,sales,dim_promotion,PRIMARY KEY,pk_dim_promotion
3,sales,dim_store,PRIMARY KEY,pk_dim_store
4,sales,fact_sales,PRIMARY KEY,pk_fact_sales
5,sales,dim_store,FOREIGN KEY,fk_store_postal_code
6,sales,fact_sales,FOREIGN KEY,fk_sales_store
7,sales,fact_sales,FOREIGN KEY,fk_sales_product
8,sales,fact_sales,FOREIGN KEY,fk_sales_promotion
9,sales,fact_sales,FOREIGN KEY,fk_sales_date


## 3. Example queries

### 3.1 Revenue, margin and growth by year
Window function `LAG` to compare each year with the previous one.

In [6]:
run_query("""
    WITH yearly AS (
        SELECT d.year,
               SUM(f.revenue) AS revenue,
               SUM(f.revenue - f.cost) AS margin
        FROM sales.fact_sales f
        JOIN sales.dim_date d ON d.date = f.date
        GROUP BY d.year
    )
    SELECT year,
           ROUND(revenue) AS revenue,
           ROUND(100 * margin / revenue, 1) AS margin_pct,
           ROUND(100 * (revenue / LAG(revenue) OVER (ORDER BY year) - 1), 1) AS growth_pct
    FROM yearly
    ORDER BY year
""")

,year,revenue,margin_pct,growth_pct
0,2021,54039295.0,42.9,NaN
1,2022,66115926.0,42.8,22.3
2,2023,83751599.0,42.8,26.7
3,2024,92737196.0,42.7,10.7
4,2025,103122866.0,42.7,11.2


### 3.2 Like-for-like growth
Growth of the stores open during the whole of both years, which removes the effect of openings and closures (a common retail metric).

In [7]:
run_query("""
    WITH store_year AS (
        SELECT f.store_id, d.year, SUM(f.revenue) AS revenue
        FROM sales.fact_sales f
        JOIN sales.dim_date d ON d.date = f.date
        GROUP BY f.store_id, d.year
    ),
    comparable AS (
        SELECT sy.year, sy.store_id, sy.revenue,
               LAG(sy.revenue) OVER (PARTITION BY sy.store_id ORDER BY sy.year) AS previous_revenue
        FROM store_year sy
        JOIN sales.dim_store s ON s.store_id = sy.store_id
        WHERE s.opening_date < make_date(sy.year - 1, 1, 1)
          AND (s.closing_date IS NULL OR s.closing_date >= make_date(sy.year, 12, 31))
    )
    SELECT year,
           COUNT(*) AS comparable_stores,
           ROUND(100 * (SUM(revenue) / SUM(previous_revenue) - 1), 1) AS like_for_like_growth_pct
    FROM comparable
    WHERE previous_revenue IS NOT NULL
    GROUP BY year
    ORDER BY year
""")

,year,comparable_stores,like_for_like_growth_pct
0,2022,32,14.6
1,2023,34,12.1
2,2024,33,7.5
3,2025,40,6.9


### 3.3 Revenue by district (Portugal)
The stores are linked to the districts through the real reference tables: store → postal code → locality → district.

In [8]:
if reference_exists:
    display(run_query("""
        SELECT dist.district_name,
               COUNT(DISTINCT s.store_id) AS stores,
               ROUND(SUM(f.revenue)) AS revenue
        FROM sales.fact_sales f
        JOIN sales.dim_store s            ON s.store_id = f.store_id
        JOIN reference.postal_codes pc    ON pc.postal_code = s.postal_code
        JOIN reference.localities l       ON l.locality_id = pc.locality_id
        JOIN reference.districts dist     ON dist.district_id = l.district_id
        GROUP BY dist.district_name
        ORDER BY revenue DESC
    """))

,district_name,stores,revenue
0,Porto,11,123941183.0
1,Lisboa,8,61270010.0
2,Aveiro,4,43292191.0
3,Setúbal,3,41912940.0
4,Coimbra,4,20065763.0
5,Braga,1,19544902.0
6,Ilha de São Miguel,1,17113966.0
7,Viana do Castelo,2,11546129.0
8,Santarém,1,2792801.0
9,Vila Real,2,1937853.0


### 3.4 Promotion effect by category
Average units per store and day, with and without promotion, in the same months (to reduce the effect of the season).

In [9]:
run_query("""
    WITH daily AS (
        SELECT p.category,
               d.month,
               f.promotion_id IS NOT NULL AS in_promotion,
               f.date, f.store_id,
               SUM(f.quantity) AS units
        FROM sales.fact_sales f
        JOIN sales.dim_product p ON p.product_id = f.product_id
        JOIN sales.dim_date d    ON d.date = f.date
        GROUP BY p.category, d.month, in_promotion, f.date, f.store_id
    ),
    promo_months AS (
        SELECT DISTINCT category, month FROM daily WHERE in_promotion
    )
    SELECT dl.category,
           ROUND(AVG(dl.units) FILTER (WHERE NOT dl.in_promotion), 2) AS units_without_promotion,
           ROUND(AVG(dl.units) FILTER (WHERE dl.in_promotion), 2) AS units_with_promotion,
           ROUND(100 * (AVG(dl.units) FILTER (WHERE dl.in_promotion)
                        / AVG(dl.units) FILTER (WHERE NOT dl.in_promotion) - 1), 1) AS change_pct
    FROM daily dl
    JOIN promo_months pm USING (category, month)
    GROUP BY dl.category
    ORDER BY change_pct DESC
""")

,category,units_without_promotion,units_with_promotion,change_pct
0,Beach & Water,13.01,34.04,161.7
1,Fitness,18.27,29.81,63.2
2,Cycling,13.91,21.05,51.4
3,Running,49.68,67.14,35.1
4,Hiking & Camping,11.48,14.78,28.8
5,Snow & Winter,17.46,20.56,17.8


### 3.5 Population within 15 km of each store (PostGIS)
`ST_DWithin` on the `geography` type finds the localities whose centre is less than 15 km from each store, and adds up their population. Only **Portuguese stores** are included, because the reference data has no population for Spain.

The generation model only uses the population of the store's **own locality** (population tier), not the population around it. This query therefore tests a different question: *does the wider catchment area also explain the revenue?* Store size and type also have a strong effect, so the correlation can be weak. Comment on the result you get.

In [10]:
if reference_exists:
    catchment = run_query("""
        WITH catchment AS (
            SELECT s.store_id,
                   SUM(l.population_2024) AS population_15km
            FROM sales.dim_store s
            JOIN reference.localities l
              ON ST_DWithin(s.geometry::geography,
                            ST_SetSRID(ST_MakePoint(l.lon_cent, l.lat_cent), 4326)::geography,
                            15000)
            GROUP BY s.store_id
        ),
        revenue AS (
            SELECT store_id, SUM(revenue) / COUNT(DISTINCT date) AS revenue_per_day
            FROM sales.fact_sales
            GROUP BY store_id
        )
        SELECT s.store_name, s.country, s.location_profile,
               c.population_15km,
               ROUND(r.revenue_per_day) AS revenue_per_day
        FROM sales.dim_store s
        JOIN catchment c USING (store_id)
        JOIN revenue r USING (store_id)
        WHERE s.country = 'PT'
        ORDER BY c.population_15km DESC
    """)
    display(catchment.head(10))
    print("Correlation (population within 15 km vs revenue per day):",
          round(catchment["population_15km"].corr(catchment["revenue_per_day"]), 2))

,store_name,country,location_profile,population_15km,revenue_per_day
0,Cume & Costa Campolide,PT,Coastal,1891468,6899.0
1,Cume & Costa Alfragide,PT,Coastal,1838599,2491.0
2,Cume & Costa Caparica e Trafaria,PT,Coastal,1747267,7501.0
3,Cume & Costa Loures,PT,Coastal,1528460,1810.0
4,Cume & Costa Santa Clara,PT,Coastal,1502069,8448.0
5,Cume & Costa Águas Santas,PT,Coastal,1070426,9022.0
6,Cume & Costa Canidelo,PT,Coastal,1069279,2368.0
7,Cume & Costa Mafamude e Vilar do Paraíso,PT,Coastal,1068663,5318.0
8,Cume & Costa Cidade da Maia,PT,Coastal,1029380,9687.0
9,Cume & Costa Carcavelos e Parede,PT,Coastal,1006871,6484.0


Correlation (population within 15 km vs revenue per day): 0.24


### 3.6 Top 10 products by gross margin

In [11]:
run_query("""
    SELECT p.product_name, p.category,
           SUM(f.quantity) AS units,
           ROUND(SUM(f.revenue)) AS revenue,
           ROUND(SUM(f.revenue - f.cost)) AS gross_margin,
           ROUND(100 * SUM(f.revenue - f.cost) / SUM(f.revenue), 1) AS margin_pct
    FROM sales.fact_sales f
    JOIN sales.dim_product p ON p.product_id = f.product_id
    GROUP BY p.product_name, p.category
    ORDER BY gross_margin DESC
    LIMIT 10
""")

,product_name,category,units,revenue,gross_margin,margin_pct
0,Road running shoes,Running,575616,55730635.0,24732692.0,44.4
1,Trail running shoes,Running,344661,40785063.0,16554710.0,40.6
2,Mountain bike,Cycling,45290,26695777.0,13835430.0,51.8
3,Hiking boots,Hiking & Camping,249807,30518078.0,11539054.0,37.8
4,Road bike,Cycling,21830,21064910.0,10949681.0,52.0
5,Running T-shirt,Running,766407,16483617.0,7065753.0,42.9
6,Wetsuit,Beach & Water,87451,13298738.0,6605344.0,49.7
7,Running shorts,Running,575069,11130954.0,5631207.0,50.6
8,Swimsuit,Beach & Water,437859,13319831.0,5111313.0,38.4
9,Surfboard,Beach & Water,26255,9296401.0,4549595.0,48.9


## Summary
* Exported the five tables to CSV and Parquet for Power BI.
* Loaded them into PostgreSQL with bulk `COPY`, created the star schema with primary and foreign keys, and linked each store to a real postal code of the reference project.
* Example queries: yearly growth and like-for-like growth with window functions, revenue by district through the reference tables, promotion effect, and the population around each store with PostGIS.